In [ ]:
# Import libraries
import csv

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

In [31]:
# Set file paths
dataset = 'model/New_english_dataset/keypoint.csv'
model_save_path = 'model/New_english_dataset/keypoint_classifier.keras'
tflite_save_path = 'model/New_english_dataset/keypoint_classifier.tflite'

In [32]:
# Set class count
NUM_CLASSES = 26

In [33]:
# Load preprocessed features
# The CSV already contains normalized 42-value landmark vectors, so we load them directly.
X_dataset = np.loadtxt(dataset, delimiter=',', dtype='float32', usecols=list(range(1, (21 * 2) + 1)))

In [34]:
# Load labels
y_dataset = np.loadtxt(dataset, delimiter=',', dtype='int32', usecols=(0))

In [35]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, train_size=0.75, random_state=RANDOM_SEED)

In [36]:
# Build classifier
# Input expects the preprocessed 42-value feature vector from the CSV.
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(42,)),  # 21*2 = 42
    
    tf.keras.layers.BatchNormalization(),
    
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    
    tf.keras.layers.Dense(32, activation='relu'),
    
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

In [37]:
# Show model summary
model.summary()  # tf.keras.utils.plot_model(model, show_shapes=True)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ batch_normalization_3           │ (None, 42)             │           168 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │         5,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 26)             │           858 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,634 (68.88 KB)

 Trainable params: 17,166 (67.05 KB)

 Non-trainable params: 468 (1.83 KB)

In [38]:
# Set callbacks
# Model checkpoint callback
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    model_save_path, verbose=1, save_weights_only=False)
# Callback for early stopping
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

In [39]:
# Compile model
# Model compilation
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [40]:
# Train model
model.fit(
    X_train,
    y_train,
    epochs=1000,
    batch_size=256,
    validation_data=(X_test, y_test),
    callbacks=[cp_callback, es_callback]
)

Epoch 1/1000
 90/109 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.2693 - loss: 2.6809
Epoch 1: saving model to model/New_english_dataset/keypoint_classifier.keras
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3084 - loss: 2.5482 - val_accuracy: 0.3993 - val_loss: 2.2162
Epoch 2/1000
106/109 ━━━━━━━━━━━━━━━━━━━━ 0s 961us/step - accuracy: 0.7935 - loss: 0.7423
Epoch 2: saving model to model/New_english_dataset/keypoint_classifier.keras
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7949 - loss: 0.7372 - val_accuracy: 0.6529 - val_loss: 1.0052
Epoch 3/1000
105/109 ━━━━━━━━━━━━━━━━━━━━ 0s 969us/step - accuracy: 0.9044 - loss: 0.3463
Epoch 3: saving model to model/New_english_dataset/keypoint_classifier.keras
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9049 - loss: 0.3445 - val_accuracy: 0.9424 - val_loss: 0.2954
Epoch 4/1000
100/109 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9360 - loss: 0.2232  
Epoch 4: saving model to model/New_english_dataset/keypoint_

In [41]:
# Evaluate model
# Model evaluation
val_loss, val_acc = model.evaluate(X_test, y_test, batch_size=128)

73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 505us/step - accuracy: 0.9975 - loss: 0.0162  


In [42]:
# Reload model
# Loading the saved model
model = tf.keras.models.load_model(model_save_path)

In [47]:
# Save inference model
# Save as a model dedicated to inference
model.save(model_save_path, include_optimizer=False)

In [48]:
# Quantize model
# Transform model (quantization)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quantized_model = converter.convert()

open(tflite_save_path, 'wb').write(tflite_quantized_model)

INFO:tensorflow:Assets written to: /var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmpvjzic7rl/assets


INFO:tensorflow:Assets written to: /var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmpvjzic7rl/assets


Saved artifact at '/var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmpvjzic7rl'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 42), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 26), dtype=tf.float32, name=None)
Captures:
  5241019200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5289024672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5290274880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5290278752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5290693728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5290696192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5290709936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5290652768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5290711168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5290712048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5290824800: TensorSpec(shap

W0000 00:00:1773590718.348600  232164 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1773590718.348763  232164 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


25720

In [49]:
# Load TFLite model
interpreter = tf.lite.Interpreter(model_path=tflite_save_path)
interpreter.allocate_tensors()

/Users/ahmad/.pyenv/versions/3.9.6/lib/python3.9/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [50]:
# Read tensor shapes
# Get I / O tensor
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [51]:
# Set input sample
interpreter.set_tensor(input_details[0]['index'], np.array([X_test[0]]))

In [52]:
%%time
# Run TFLite inference
# Inference implementation
interpreter.invoke()
tflite_results = interpreter.get_tensor(output_details[0]['index'])

CPU times: user 168 µs, sys: 98 µs, total: 266 µs
Wall time: 497 µs


In [53]:
# Print predictions
print(np.squeeze(tflite_results))
print(np.argmax(np.squeeze(tflite_results)))

[3.79713316e-10 2.59807124e-11 2.01147254e-09 7.54598328e-11
 7.49461948e-18 1.00925095e-12 2.35201885e-08 2.62093515e-12
 3.99963847e-11 3.99509982e-12 9.41671452e-09 1.34712137e-11
 1.00000000e+00 4.39432313e-09 4.26974862e-12 2.66717963e-12
 4.00167177e-13 1.64072523e-11 2.36267453e-16 3.57215906e-12
 1.94122309e-16 3.12273059e-12 1.42926565e-11 1.47159163e-16
 4.34540614e-13 6.52893419e-15]
12


In [ ]:
# Empty cell


In [ ]:
# Empty cell
